In [10]:
# 必要なモジュールをインポート
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import ChatCompletionToolParam
from tavily import TavilyClient

# 環境変数の取得
load_dotenv("../.env")

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# tavily検索用APIキーの取得
TAVILY_API_KEY = os.environ['TAVILY_API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini"

# 検索結果を返す関数の作成
def get_search_result(question):
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(question)
    return json.dumps({"result": response["results"]})

# ツール定義
def define_tools():
    print("------define_tools(ツール定義)------")
    return [
        ChatCompletionToolParam({
            "type": "function",
            "function": {
                "name": "get_search_result",
                "description": "最近一ヵ月のイベント開催予定などネット検索が必要な場合に、質問文の検索結果を取得する",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "question": {"type": "string", "description": "質問文"},
                    },
                    "required": ["question"],
                },
            },
        })
    ]

def ask_question(messages, tools):
    return client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )

# ツール呼び出しが必要な場合の処理を行う関数
def handle_tool_call(response, messages, tools):
    # 関数の実行と結果取得
    tool = response.choices[0].message.tool_calls[0]
    function_name = tool.function.name
    arguments = json.loads(tool.function.arguments)
    function_response = globals()[function_name](**arguments)

    # assistantのtool_callメッセージを履歴に追加
    messages.append(response.choices[0].message)

    # tool結果を履歴に追加
    messages.append({
        "role": "tool",
        "tool_call_id": tool.id,
        "content": function_response,
    })

    # 関数の実行結果をmessagesに加えて再度言語モデルを呼出
    return client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

# ユーザーからの質問を処理する関数
def process_response(messages, tools):
    response = ask_question(messages, tools)

    if response.choices[0].finish_reason == 'tool_calls':
        # ツール呼出の場合
        response = handle_tool_call(response, messages, tools)
    return response.choices[0].message.content.strip()

# キャラクター設定を行う関数
def character_setting():
    role_message = input("キャラクターの設定: ")
    if role_message:
        messages.append({"role": "system", "content": role_message})


tools = define_tools()

messages=[]

character_setting()

while(True):
    # ユーザーからの質問を受付
    question = input("メッセージを入力:")
    # 質問が入力されなければ終了
    if question.strip()=="":
        break
    display(f"質問:{question}")

    # メッセージにユーザーからの質問を追加
    messages.append({"role": "user", "content": question.strip()})
    # やりとりが8を超えたら古いメッセージから削除
    if len(messages) > 8:
        del_message = messages.pop(0)

    # 言語モデルに質問
    response_message = process_response(messages, tools)

    # メッセージに言語モデルからの回答を追加
    print(response_message, flush=True)
    messages.append({"role": "assistant", "content": response_message})

print("\n---ご利用ありがとうございました！---")


------define_tools(ツール定義)------


'質問:こんにちは'

こんにちはにゃ！今日はどんなことをお話ししたいにゃ？


'質問:東北6県は？'

東北6県は、以下の県にゃ！

1. 青森県（あおもりけん）にゃ
2. 岩手県（いわてけん）にゃ
3. 宮城県（みやぎけん）にゃ
4. 秋田県（あきたけん）にゃ
5. 山形県（やまがたけん）にゃ
6. 福島県（ふくしまけん）にゃ

何か他に知りたいことがあれば教えてにゃ！


...

宮城県のお土産についての情報をいくつか紹介するにゃ！

1. **[宮城県産の特産品やお土産一覧](https://shunsentanbou.pref.miyagi.jp/theme/miyage/)**  
   ここでは、宮城県の人気のお土産を紹介しているにゃ！例えば、名物の「大橋の豆腐」や「生クリーム大福」などがあるにゃ。

2. **[宮城県のお土産特集](https://food.biglobe.ne.jp/prefectures_tohoku_miyagi/scene_souvenir/cate_food/)**  
   宮城県の特産品である「松島のホタテ」や「仙台牛」を使ったお土産も有名にゃ。他にも「ずんだ餅」などのスイーツも人気にゃよ。

3. **[おすすめの名物お土産](https://tsplus.asahi.co.jp/articles/gift/75865/)**  
   ここでは、特に人気のあるお土産やその特徴を詳しく解説しているにゃ。仙台の「牛たん」や「ずんだ」といった名物が紹介されているにゃよ！

4. **[宮城県物産協会のオンラインショップ](https://ec.miyagibussan.or.jp/)**  
   宮城の特産品をオンラインで購入できるサイトもあるにゃ。手軽に宮城の特産品を楽しめるにゃよ！

他にも知りたいことがあれば教えてにゃ！


'質問:福島県のお土産について検索した結果を教えて'

福島県のお土産についての情報をいくつか紹介するにゃ！

1. **[おすすめの福島のお土産10選](https://macaro-ni.jp/159948)**  
   これでは、福島で手に入れたいお土産が紹介されているにゃ。例えば、「ままどおる」や「喜多方ラーメン」など人気の商品があるにゃよ。

2. **[福島県産のお土産情報](https://food.biglobe.ne.jp/prefectures_tohoku_fukushima/scene_souvenir/cate_food/)**  
   福島県特産の「喜多方ラーメン」のほか、「福島牛」や「ふくしまの地酒」などの情報が載っているにゃ。

3. **[福島県のお土産特集動画](https://www.youtube.com/watch?v=KvDtshVt20M)**  
   福島県でのおすすめお土産を集めた動画もあるにゃ。視覚的に見ることで選びやすいにゃね。

4. **[福島県の観光物産館](https://www.tif.ne.jp/bussan/bussankan/gift/index.html)**  
   こちらでは福島県の様々なお土産が一覧で紹介されており、地元の名産品を簡単に探せるにゃよ！

5. **[郡山の現地情報](https://s.tabelog.com/smartphone/matome/27685/)**  
   こちらは郡山で購入できるお土産の紹介となっており、特に人気の商品が多く載っているにゃ。

何か他に知りたいことがあれば教えてにゃ！


'質問:宮城県の観光について教えて'

宮城県の観光スポットについての情報をいくつか紹介するにゃ！

1. **[宮城県の観光スポット - JTB](https://www.jtb.co.jp/kokunai-guide/list/index.html?area=04&feature=0&genre=000)**  
   ここでは、宮城県の観光名所が一覧で紹介されていて、松島や秋保温泉、仙台の文化施設など、多彩なスポットが見つかるにゃよ。

2. **[松島の観光スポットをおすすめ！](https://article.his-j.com/kokunai/tohoku/miyagi/post-14585/)**  
   松島は福島三景の一つとして有名で、観光名所がたくさんあるにゃ。特に松島湾の絶景や遊覧船が楽しめるにゃよ。

3. **[観光地の魅力を集めた特集](https://event.jreast.co.jp/pages/miyagi_886)**  
   ここでは、宮城県の各観光地の魅力を詳しく紹介していて、もちろん松島や仙台なども含まれているにゃ！

4. **[宮城の観光についてのガイド](https://www.hankyu-travel.com/guide/tohoku/miyagi/)**  
   こちらでは、宮城県の観光名所や地域の人々の魅力を紹介しているにゃ。ローカルな体験もできるにゃよ。

5. **[2025年版 宮城観光スポットランキング](https://travel.rakuten.co.jp/mytrip/ranking/spot-miyagi)**  
   新しい観光スポットのランキングが紹介されていて、2025年に行ってみたい場所がわかるにゃよ。

これらの情報を参考に、宮城県の観光を楽しんでにゃ！他に知りたいことがあれば教えてにゃ！

---ご利用ありがとうございました！---
